In [3]:
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb

util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Using real_world_data_jun_2022 ....
Spark Job Progress Monitor already enabled


In [4]:
df_or = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/dataframe030723')

In [24]:
df_or.columns

▸,:,


['personid',
 'birthdate',
 'gender',
 'race',
 'conditioncode',
 'diagnosis_date',
 'age_at_diagnosis',
 'nI10',
 'nE11_9',
 'nE78_5',
 'nZ79_899',
 'nS09_90XA',
 'nZ23',
 'nI25_10',
 'nK21_9',
 'nE03_9',
 'nF41_9',
 'nZ79_01',
 'nF32_9',
 'nZ87_891',
 'nI48_91',
 'nF17_210',
 'nR10_9',
 'nN39_0',
 'nZ79_82',
 'nD64_9',
 'nR05',
 'EPI']

In [25]:
df_data = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/epilepsy_lab_new.parquet')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
##import org.apache.spark.sql.SparkSession
##import org.apache.spark.sql.functions._
records = df_data.filter((df_data.personid)=='0103bf48-dc20-438c-b3d4-08dcd018440f').show(truncate=False)

+------------------------------------+------------------------------------+------------------------------------+-------+--------------------------------------------------------+----------+-------------------------+-----+-----------------+------+-------------------------+---+
|labid                               |encounterid                         |personid                            |labcode|primaryDisplay                                          |loincclass|servicedate              |value|interpretation   |source|diagnosis_date           |EPI|
+------------------------------------+------------------------------------+------------------------------------+-------+--------------------------------------------------------+----------+-------------------------+-----+-----------------+------+-------------------------+---+
|cb269bfd-34a3-4301-8344-545b7df41b68|dc6a4d85-3557-43f7-9c86-57a5c8714790|0103bf48-dc20-438c-b3d4-08dcd018440f|787-2  |MCV [Entitic volume] by Automated count             

In [7]:
df_data.printSchema()

root
 |-- labid: string (nullable = true)
 |-- encounterid: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- primaryDisplay: string (nullable = true)
 |-- loincclass: string (nullable = true)
 |-- servicedate: string (nullable = true)
 |-- value: string (nullable = true)
 |-- interpretation: string (nullable = true)
 |-- source: string (nullable = true)
 |-- diagnosis_date: string (nullable = true)
 |-- EPI: integer (nullable = true)



In [4]:
from pyspark.sql.functions import *
def process_lab_test(spark, cohort, lab):
    cohort.createOrReplaceTempView('cohort_temp')
    lab.createOrReplaceTempView('lab_temp')

    lab_processed = spark.sql("""
WITH temp1 AS (
    SELECT t.personid, 
           DATE_ADD(MIN(t.diagnosis_date), -180) AS date_before_six_months,
           DATE_ADD(MIN(t.diagnosis_date), -270) as date_before_two_years
    FROM cohort_temp t
    GROUP BY t.personid
)
SELECT t1.personid, 
       l.labcode, 
       avg(l.value)
FROM temp1 t1
INNER JOIN lab_temp l ON t1.personid = l.personid
WHERE l.servicedate between t1.date_before_two_years and t1.date_before_six_months 
group by t1.personid,l.labcode
    """)    
    ##lab_processed = lab_processed.groupBy('personid').pivot('labcode').agg(avg('value'))
    #WHERE l.servicedate <= date_add(MIN(date), -6*30)
    #WHERE t1.latest_servicedate BETWEEN t1.date_before_six_months AND t1.earliest_date_of_diagnosis
    return lab_processed

In [6]:
lab_processed = process_lab_test(spark, df, df_data)

In [8]:
lab_processed.show(5, truncate = False)

+------------------------------------+-------+--------------------------+
|personid                            |labcode|avg(CAST(value AS DOUBLE))|
+------------------------------------+-------+--------------------------+
|0103bf48-dc20-438c-b3d4-08dcd018440f|787-2  |100.0                     |
|0103bf48-dc20-438c-b3d4-08dcd018440f|718-7  |13.7                      |
|0103bf48-dc20-438c-b3d4-08dcd018440f|6690-2 |5.1                       |
|0103bf48-dc20-438c-b3d4-08dcd018440f|1751-7 |4.4                       |
|0103bf48-dc20-438c-b3d4-08dcd018440f|789-8  |3.89                      |
+------------------------------------+-------+--------------------------+
only showing top 5 rows



In [28]:
from pyspark.sql.functions import *
joined_filtered = df_data.filter((months_between(col('diagnosis_date'),col('servicedate')) >= 6) & (months_between(col('diagnosis_date'),col('servicedate')) <= 24))

▸,:,


In [29]:
joined_filtered_small = joined_filtered.\
    select('personid','labcode','value','servicedate').distinct().\
    groupBy('personid','labcode','servicedate').agg(avg('value').alias('avg_value'))\
    .select('personid','labcode','avg_value').groupBy('personid','labcode').agg(avg('avg_value').alias('avg_v'))

▸,:,


In [17]:
joined_filtered.filter((joined_filtered.personid)=='0103bf48-dc20-438c-b3d4-08dcd018440f').show(truncate=False)

+------------------------------------+------------------------------------+------------------------------------+-------+--------------------------------------------------------+----------+-------------------------+-----+-----------------+------+-------------------------+---+
|labid                               |encounterid                         |personid                            |labcode|primaryDisplay                                          |loincclass|servicedate              |value|interpretation   |source|diagnosis_date           |EPI|
+------------------------------------+------------------------------------+------------------------------------+-------+--------------------------------------------------------+----------+-------------------------+-----+-----------------+------+-------------------------+---+
|b306a93f-0725-44a6-a6e6-ad5c3d629143|e86329e0-79c4-4bb7-a1d0-75e17cf1ea3b|0103bf48-dc20-438c-b3d4-08dcd018440f|787-2  |MCV [Entitic volume] by Automated count             

In [30]:
joined_filtered_pivot = joined_filtered_small.groupBy('personid').pivot('labcode').agg(avg('avg_v'))

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
joined_filtered_pivot.columns

▸,:,


<IPython.core.display.Javascript object>

['personid',
 '16325-3',
 '1751-7',
 '28539-5',
 '28540-3',
 '3043-7',
 '34543-9',
 '38483-4',
 '41276-7',
 '4544-3',
 '4548-4',
 '49765-1',
 '6690-2',
 '718-7',
 '77138-6',
 '787-2',
 '789-8']

In [32]:
joined_filtered_pivot.show(5, truncate = False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

KeyboardInterrupt: 

In [ ]:
df_or.createOrReplaceTempView('df_or')
joined_filtered_pivot.createOrReplaceTempView('joined_filtered_pivot')
df_check = spark.sql("""
SELECT
    df_or.personid,
    df_or.birthdate,
    df_or.gender,
    df_or.race,
    df_or.conditioncode,
    df_or.diagnosis_date,
    df_or.age_at_diagnosis,
    df_or.nI10,
    df_or.nE11_9,
    df_or.nE78_5,
    df_or.nZ79_899,
    df_or.nS09_90XA,
    df_or.nZ23,
    df_or.nI25_10,
    df_or.nK21_9,
    df_or.nE03_9,
    df_or.nF41_9,
    df_or.nZ79_01,
    df_or.nF32_9,
    df_or.nZ87_891,
    df_or.nI48_91,
    df_or.nF17_210,
    df_or.nR10_9,
    df_or.nN39_0,
    df_or.nZ79_82,
    df_or.nD64_9,
    df_or.nR05,
    df_or.EPI,
    a.`16325-3`,
    a.`1751-7`,
    a.`28539-5`,
    a.`28540-3`,
    a.`3043-7`,
    a.`34543-9`,
    a.`38483-4`,
    a.`41276-7`,
    a.`4544-3`,
    a.`4548-4`,
    a.`49765-1`,
    a.`6690-2`,
    a.`718-7`,
    a.`77138-6`,
    a.`787-2`,
    a.`789-8`

FROM df_or 
INNER JOIN joined_filtered_pivot a
ON df_or.personid = a.personid
""")


df_check.show(10, truncate  = False)

In [83]:
df_check.write.parquet('epilepsy_pred_draft_sa.parquet')

In [85]:
df_check.write.parquet('file:/home/o_suchsi/work/Oklahoma State/Sai/epilepsy_pred_draft_sa')

In [84]:
df_check.count()

1582021

In [56]:
df_check.createOrReplaceTempView('df_check')
##e_result = spark.sql("""SELECT df_check.personid., COUNT(df_check.personid.) FROM df_check  where EPI=1 group by df_check.personid. having COUNT(df_check.personid. > 1)
 ##               """)
result = df_check.filter(col("EPI") == 1) \
    .groupBy("personid") \
    .agg({"personid": "count"}) \
    .withColumnRenamed("count(personid)", "count_personid") \
    .filter(col("count_personid") > 1)
result.show(10, truncate  = False)

AnalysisException: "Reference 'personid' is ambiguous, could be: or.personid, pi.personid.;"

In [28]:
df_final = df_or.join(joined_filtered_pivot, df_or.personid == joined_filtered_pivot.personid, 'inner')

In [43]:
df_final.createOrReplaceTempView('df_final')
e_result = spark.sql("""(SELECT d.personid, COUNT(d.personid) FROM df_final d where EPI=1 group by personid having COUNT(personid)>1)
                """)

e_result.show(10, truncate  = False)

AnalysisException: "Reference 'personid' is ambiguous, could be: d.personid, d.personid.; line 1 pos 75"

In [40]:
df_final.write.parquet('epilepsy_pred_draft.parquet')

AnalysisException: 'Found duplicate column(s) when inserting into hdfs://ip-10-42-48-233.us-east-2.compute.internal:8020/user/o_suchsi/epilepsy_pred_draft.parquet: `personid`;'

In [38]:
df_final.count()

1582021

In [33]:
df_final.printSchema()

root
 |-- personid: string (nullable = true)
 |-- birthdate: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- race: string (nullable = true)
 |-- conditioncode: string (nullable = true)
 |-- diagnosis_date: string (nullable = true)
 |-- age_at_diagnosis: double (nullable = true)
 |-- nI10: long (nullable = true)
 |-- nE11_9: long (nullable = true)
 |-- nE78_5: long (nullable = true)
 |-- nZ79_899: long (nullable = true)
 |-- nS09_90XA: long (nullable = true)
 |-- nZ23: long (nullable = true)
 |-- nI25_10: long (nullable = true)
 |-- nK21_9: long (nullable = true)
 |-- nE03_9: long (nullable = true)
 |-- nF41_9: long (nullable = true)
 |-- nZ79_01: long (nullable = true)
 |-- nF32_9: long (nullable = true)
 |-- nZ87_891: long (nullable = true)
 |-- nI48_91: long (nullable = true)
 |-- nF17_210: long (nullable = true)
 |-- nR10_9: long (nullable = true)
 |-- nN39_0: long (nullable = true)
 |-- nZ79_82: long (nullable = true)
 |-- nD64_9: long (nullable = true)
 |-- nR0

In [36]:
df_final.createOrReplaceTempView('d_final')
check = spark.sql("""
select * from d_final d
where d.personid = '0103bf48-dc20-438c-b3d4-08dcd018440f'""")
check.show(10, truncate = False)

AnalysisException: "Reference 'd.personid' is ambiguous, could be: d.personid, d.personid.; line 3 pos 6"

In [30]:
df_final.count()

KeyboardInterrupt: 

In [49]:
s_df = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Sai/epilepsy_pred_draft_sa')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
check.createOrReplaceTempView('s_df')
dup_check = spark.sql("""
SELECT d.personid, COUNT(d.personid) FROM s_df d where EPI=1 group by personid having COUNT(personid)>1
                """)
dup_check.show(10, truncate=False)


▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------+---------------+
|personid|count(personid)|
+--------+---------------+
+--------+---------------+



<IPython.core.display.Javascript object>

In [16]:
check.createOrReplaceTempView('epilepsy_lab_result_modified_sai')
check = spark.sql("""
select * from epilepsy_lab_result_modified_sai 
where personid = '29842178-98ff-4bf9-8547-3159fb5b2dc5'""")
check.show(30, truncate = False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------------+-----+-------------+------------------+----+----+------+------+----+-------+------+------+--------+---------+--------+------+------+-------+------+--------+-------+------+-------+------+-------+------+-------+-------+------+-------+-------+-------+------+------+-------+------+-----+-------+-----+-----+---+------+
|personid                            |diagnosis_date     |race |conditioncode|age_at_diagnosis  |nR05|nI10|nE11_9|nE78_5|nZ23|nI25_10|nK21_9|nE03_9|nZ79_899|nS09_90XA|nF17_210|nR10_9|nF41_9|nZ79_01|nF32_9|nZ87_891|nI48_91|nN39_0|nZ79_82|nD64_9|16325_3|1751_7|28539_5|28540_3|3043_7|34543_9|38483_4|41276_7|4544_3|4548_4|49765_1|6690_2|718_7|77138_6|787_2|789_8|EPI|gender|
+------------------------------------+-------------------+-----+-------------+------------------+----+----+------+------+----+-------+------+------+--------+---------+--------+------+------+-------+------+--------+-------+------+-------+------+-------+--

<IPython.core.display.Javascript object>

In [51]:
epilepsy_lab_result_modified_sai = spark.sql("""
   with cte as(select s_df.personid, min(s_df.age_at_diagnosis) as age_at_diagnosis from s_df group by personid)
   SELECT r.personid, r.diagnosis_date, r.gender, r.race, r.conditioncode, r.age_at_diagnosis, r.nR05, r.nI10, r.nE11_9, r.nE78_5, r.nZ23, r.nI25_10, r.nK21_9, r.nE03_9, r.nZ79_899, r.nS09_90XA, r.nF17_210, r.nR10_9,
    r.nF41_9, r.nZ79_01, r.nF32_9, r.nZ87_891, r.nI48_91, r.nN39_0, r.nZ79_82, r.nD64_9, r.`16325_3`, r.`1751_7`, r.`28539_5`, r.`28540_3`, r.`3043_7`,
    r.`34543_9`, r.`38483_4`, r.`41276_7`, r.`4544_3`, r.`4548_4`, r.`49765_1`, r.`6690_2`, r.`718_7`, r.`77138_6`, r.`787_2`, r.`789_8`, r.EPI
    FROM s_df r
    INNER JOIN cte c
    ON r.personid = c.personid
 and r.age_at_diagnosis = c.age_at_diagnosis
 """)
epilepsy_lab_result_modified_sai.count()

▸,:,


<IPython.core.display.Javascript object>

Py4JJavaError: An error occurred while calling o822.count.
: org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:226)
	at org.apache.spark.sql.execution.exchange.BroadcastExchangeExec.doExecuteBroadcast(BroadcastExchangeExec.scala:146)
	at org.apache.spark.sql.execution.InputAdapter.doExecuteBroadcast(WholeStageCodegenExec.scala:387)
	at org.apache.spark.sql.execution.SparkPlan$$anonfun$executeBroadcast$1.apply(SparkPlan.scala:144)
	at org.apache.spark.sql.execution.SparkPlan$$anonfun$executeBroadcast$1.apply(SparkPlan.scala:140)
	at org.apache.spark.sql.execution.SparkPlan$$anonfun$executeQuery$1.apply(SparkPlan.scala:155)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:151)
	at org.apache.spark.sql.execution.SparkPlan.executeQuery(SparkPlan.scala:152)
	at org.apache.spark.sql.execution.SparkPlan.executeBroadcast(SparkPlan.scala:140)
	at org.apache.spark.sql.execution.join

<IPython.core.display.Javascript object>

In [31]:
epilepsy_lab_result_modified_sai.write.parquet('epilepsy_pred_08_14_23.parquet')

In [32]:
!hadoop fs -copyToLocal epilepsy_pred_08_14_23.parquet /home/o_suchsi/work/Oklahoma%20State/Priya/epilepsy

In [18]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType

schema = StructType([
    StructField("personid", StringType(), nullable=False),
    StructField("gender", StringType(), nullable=True),
    StructField("diagnosis_date", StringType(), nullable=True),
    StructField("race", StringType(), nullable=True),
    StructField("conditioncode", StringType(), nullable=True),
    StructField("age_at_diagnosis", DoubleType(), nullable=True),
    StructField("nR05", LongType(), nullable=True),
    StructField("nI10", LongType(), nullable=True),
    StructField("nE11_9", LongType(), nullable=True),
    StructField("nE78_5", LongType(), nullable=True),
    StructField("nZ23", LongType(), nullable=True),
    StructField("nI25_10", LongType(), nullable=True),
    StructField("nK21_9", LongType(), nullable=True),
    StructField("nE03_9", LongType(), nullable=True),
    StructField("nZ79_899", LongType(), nullable=True),
    StructField("nS09_90XA", LongType(), nullable=True),
    StructField("nF17_210", LongType(), nullable=True),
    StructField("nR10_9", LongType(), nullable=True),
    StructField("nF41_9", LongType(), nullable=True),
    StructField("nZ79_01", LongType(), nullable=True),
    StructField("nF32_9", LongType(), nullable=True),
    StructField("nZ87_891", LongType(), nullable=True),
    StructField("nI48_91", LongType(), nullable=True),
    StructField("nN39_0", LongType(), nullable=True),
    StructField("nZ79_82", LongType(), nullable=True),
    StructField("nD64_9", LongType(), nullable=True),
    StructField("34543_9", DoubleType(), nullable=True),
    StructField("38483_4", DoubleType(), nullable=True),
    StructField("41276_7", DoubleType(), nullable=True),
    StructField("4544_3", DoubleType(), nullable=True),
    StructField("4548_4", DoubleType(), nullable=True),
    StructField("49765_1", DoubleType(), nullable=True),
    StructField("6690_2", DoubleType(), nullable=True),
    StructField("718_7", DoubleType(), nullable=True),
    StructField("77138_6", DoubleType(), nullable=True),
    StructField("787_2", DoubleType(), nullable=True),
    StructField("789_8", DoubleType(), nullable=True),
    StructField("EPI", DoubleType(), nullable=True),
    StructField("16325_3", DoubleType(), nullable=True),
    StructField("1751_7", DoubleType(), nullable=True),
    StructField("28539_5", DoubleType(), nullable=True),
    StructField("28540_3", DoubleType(), nullable=True),
    StructField("3043_7", DoubleType(), nullable=True)

])

▸,:,


In [5]:
!pwd

▸,:,


/home/o_suchsi/work/Oklahoma State/Priya/epilepsy


In [1]:
s_df_p = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/epilepsy_pred_08_14_23.parquet')

In [21]:
s_df_p.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- diagnosis_date: string (nullable = true)
 |-- race: string (nullable = true)
 |-- conditioncode: string (nullable = true)
 |-- age_at_diagnosis: double (nullable = true)
 |-- nR05: long (nullable = true)
 |-- nI10: long (nullable = true)
 |-- nE11_9: long (nullable = true)
 |-- nE78_5: long (nullable = true)
 |-- nZ23: long (nullable = true)
 |-- nI25_10: long (nullable = true)
 |-- nK21_9: long (nullable = true)
 |-- nE03_9: long (nullable = true)
 |-- nZ79_899: long (nullable = true)
 |-- nS09_90XA: long (nullable = true)
 |-- nF17_210: long (nullable = true)
 |-- nR10_9: long (nullable = true)
 |-- nF41_9: long (nullable = true)
 |-- nZ79_01: long (nullable = true)
 |-- nF32_9: long (nullable = true)
 |-- nZ87_891: long (nullable = true)
 |-- nI48_91: long (nullable = true)
 |-- nN39_0: long (nullable = true)
 |-- nZ79_82: long (nullable = true)
 |-- nD64_9: long (nullable = true)
 |-- 34543_9: d

In [7]:
s_df_p.count()

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

476852

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
s_df_p.createOrReplaceTempView('epilepsy_lab_result_modified_sai')
check = spark.sql("""
select r.personid,r.diagnosis_date, r.race, r.conditioncode, r.age_at_diagnosis, r.nR05, r.nI10, r.nE11_9, r.nE78_5, r.nZ23, r.nI25_10, r.nK21_9, r.nE03_9, r.nZ79_899, r.nS09_90XA, r.nF17_210, r.nR10_9,
    r.nF41_9, r.nZ79_01, r.nF32_9, r.nZ87_891, r.nI48_91, r.nN39_0, r.nZ79_82, r.nD64_9, r.`16325_3`, r.`1751_7`, r.`28539_5`, r.`28540_3`, r.`3043_7`,
    r.`34543_9`, r.`38483_4`, r.`41276_7`, r.`4544_3`, r.`4548_4`, r.`49765_1`, r.`6690_2`, r.`718_7`, r.`77138_6`, r.`787_2`, r.`789_8`, r.EPI,
case
when (sum(case when gender = 'Male' then 1 else 0 end)) > 0 then 'Male' 
when (sum(case when gender = 'Female' then 1 else 0 end)) > 0 then 'Female'
else 'other_gender' 
end as gender
from epilepsy_lab_result_modified_sai r
group by r.personid,r.diagnosis_date, r.race, r.conditioncode, r.age_at_diagnosis, r.nR05, r.nI10, r.nE11_9, r.nE78_5, r.nZ23, r.nI25_10, r.nK21_9, r.nE03_9, r.nZ79_899, r.nS09_90XA, r.nF17_210, r.nR10_9,
    r.nF41_9, r.nZ79_01, r.nF32_9, r.nZ87_891, r.nI48_91, r.nN39_0, r.nZ79_82, r.nD64_9, r.`16325_3`, r.`1751_7`, r.`28539_5`, r.`28540_3`, r.`3043_7`,
    r.`34543_9`, r.`38483_4`, r.`41276_7`, r.`4544_3`, r.`4548_4`, r.`49765_1`, r.`6690_2`, r.`718_7`, r.`77138_6`, r.`787_2`, r.`789_8`, r.EPI
""")
check.show(30, truncate = False)
check.count()

+------------------------------------+-------------------------+----------+-------------+------------------+----+----+------+------+----+-------+------+------+--------+---------+--------+------+------+-------+------+--------+-------+------+-------+------+-------+------------------+-------+-------+------+-------+-------+------------------+------------------+------+-------+------------------+------------------+-------+------------------+------------------+---+------+
|personid                            |diagnosis_date           |race      |conditioncode|age_at_diagnosis  |nR05|nI10|nE11_9|nE78_5|nZ23|nI25_10|nK21_9|nE03_9|nZ79_899|nS09_90XA|nF17_210|nR10_9|nF41_9|nZ79_01|nF32_9|nZ87_891|nI48_91|nN39_0|nZ79_82|nD64_9|16325_3|1751_7            |28539_5|28540_3|3043_7|34543_9|38483_4|41276_7           |4544_3            |4548_4|49765_1|6690_2            |718_7             |77138_6|787_2             |789_8             |EPI|gender|
+------------------------------------+----------------------

474471

In [4]:

check.write.parquet('file:/home/o_suchsi/work/Oklahoma State/Sai/epilepsy_pred_08_16_23.parquet')

In [5]:
result_df = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Sai/epilepsy_pred_08_16_23.parquet')

In [6]:
result_df.printSchema()

root
 |-- personid: string (nullable = true)
 |-- diagnosis_date: string (nullable = true)
 |-- race: string (nullable = true)
 |-- conditioncode: string (nullable = true)
 |-- age_at_diagnosis: double (nullable = true)
 |-- nR05: long (nullable = true)
 |-- nI10: long (nullable = true)
 |-- nE11_9: long (nullable = true)
 |-- nE78_5: long (nullable = true)
 |-- nZ23: long (nullable = true)
 |-- nI25_10: long (nullable = true)
 |-- nK21_9: long (nullable = true)
 |-- nE03_9: long (nullable = true)
 |-- nZ79_899: long (nullable = true)
 |-- nS09_90XA: long (nullable = true)
 |-- nF17_210: long (nullable = true)
 |-- nR10_9: long (nullable = true)
 |-- nF41_9: long (nullable = true)
 |-- nZ79_01: long (nullable = true)
 |-- nF32_9: long (nullable = true)
 |-- nZ87_891: long (nullable = true)
 |-- nI48_91: long (nullable = true)
 |-- nN39_0: long (nullable = true)
 |-- nZ79_82: long (nullable = true)
 |-- nD64_9: long (nullable = true)
 |-- 16325_3: double (nullable = true)
 |-- 1751_7: d

In [7]:
result_df.show(5)

+--------------------+--------------------+----------+-------------+------------------+----+----+------+------+----+-------+------+------+--------+---------+--------+------+------+-------+------+--------+-------+------+-------+------+-------+------+-------+-------+------+-------+-------+-------+------------------+------+-------+------------------+-----+-------+-----+-----------------+---+------+
|            personid|      diagnosis_date|      race|conditioncode|  age_at_diagnosis|nR05|nI10|nE11_9|nE78_5|nZ23|nI25_10|nK21_9|nE03_9|nZ79_899|nS09_90XA|nF17_210|nR10_9|nF41_9|nZ79_01|nF32_9|nZ87_891|nI48_91|nN39_0|nZ79_82|nD64_9|16325_3|1751_7|28539_5|28540_3|3043_7|34543_9|38483_4|41276_7|            4544_3|4548_4|49765_1|            6690_2|718_7|77138_6|787_2|            789_8|EPI|gender|
+--------------------+--------------------+----------+-------------+------------------+----+----+------+------+----+-------+------+------+--------+---------+--------+------+------+-------+------+-------

In [13]:
check.createOrReplaceTempView('check')
a=spark.sql("""
select count(personid) from check
where gender = 'Female' """)
a.show(10)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+---------------+
|count(personid)|
+---------------+
|         264613|
+---------------+



<IPython.core.display.Javascript object>

In [5]:
dup_check.show(10, truncate= False)

+------------------------------------+---------------+
|personid                            |count(personid)|
+------------------------------------+---------------+
|001e4ea5-2b81-4093-a5f2-228c5f56bd19|12             |
|0123b1d3-6f2f-4dbd-abaf-bbfc46ede169|2              |
|0137680e-a2a0-4531-bdff-f43052865b70|4              |
|0185fdd7-4c5b-451e-a41c-3b8c690eb9b3|14             |
|01d7953f-b4c8-404f-99d9-06a7ece438e2|2              |
|022c889c-3a19-487c-924a-e17af130d0c2|2              |
|025ff86d-bea8-4525-a397-9b00f3da5178|23             |
|030160ee-4cda-49f1-8bc1-2fb7460cef4d|6              |
|031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5|2              |
|032145cc-8b8d-41b0-8b15-198fa7c95a68|12             |
+------------------------------------+---------------+
only showing top 10 rows

